In [5]:
import pandas as pd

In [ ]:

def process_county_state_dataset(file_path):
    
    df = pd.read_csv(file_path, low_memory=False)
    
    # Drop columns with `_se`, `_n`, `par_rank_[race]_[gender]_mean`, and `frac_years_xw_[race]_[gender]`
    df = df[df.columns.drop(list(df.filter(regex='_se$|_n$|par_rank_.*_mean$|frac_years_xw_.*')))]
    
    # Filter to include only the specified outcomes using string slicing
    outcomes_to_keep = [
        'coll', 'hours_wk', 'hs', 'jail', 'kir', 'kir_top01', 'kir_top20', 
        'kir_24', 'kir_26', 'kir_29', 'lpov_nbh', 'proginc', 'somecoll', 
        'staytract', 'teenbrth', 'wgflx_rk', 'working',
    ]
    outcomes_to_exclude = ['kir_imm','kir_native','kir_stycz']
    
    valid_races = {'white','black','asian','hisp','natam','other','pooled'}
    
    # Defines a function to check if a column belongs to the specified outcomes

    def is_desired_outcome(col):
        # split and find the first race token, everything before it is the outcome
        parts = col.split('_')
        race_pos = None
        for i, p in enumerate(parts):
            if p in valid_races:
                race_pos = i
                break
        if race_pos is None:
            return False  # doesn't match the expected pattern

        outcome_token = '_'.join(parts[:race_pos])
        if outcome_token in outcomes_to_exclude:
            return False
        return outcome_token in outcomes_to_keep
        
    # Filter columns using the string slicing method
    base_cols = ['state', 'county', 'cz', 'czname']
    filtered_cols = [col for col in df.columns if is_desired_outcome(col)] # creates a new list of columns to keep based on the function above

    # print([c for c in filtered_cols if c.startswith("kir_")]) , debugging

    df = df[base_cols + filtered_cols] # re-subsets dataframe for all columns we want to keep
    
    # Melt the DataFrame to long format
    df_long = df.melt(id_vars=base_cols, var_name='category', value_name='value')
    
    # Extract components and assign `value_type`
    pattern = (
        r'(?P<outcome>\w+)_(?P<race>white|black|asian|hisp|natam|other|pooled)_'  # Matches outcome and race
        r'(?P<gender>male|female|pooled)'                                          # Matches gender
        r'(?:_p(?P<percentile>\d+))?'                                              # Matches percentile if present
    )
    df_long[['outcome', 'race', 'gender', 'percentile']] = df_long['category'].str.extract(pattern) # looks at each string in category column, pulls out outcome, race, gender, pctile in each row using 
    #.str.extract, assigns vals into new dataframe columns of outcome, race, gender, percentile
    
    # Assign value_type based on column naming convention
    def assign_value_type(category):
        if '_mean' in category:
            return 'mean'
        else:
            return 'distinct'
    
    df_long['value_type'] = df_long['category'].apply(assign_value_type)
    
    # Fill NaN for percentiles in mean values with null
    df_long.loc[df_long['value_type'] == 'mean', 'percentile'] = None
   
    
    # Remove unneeded columns and sort
    df_long.drop(columns=['category'], inplace=True)

    # Rename outcomes
    outcome_labels = {
        'kir': 'Income Rank (child earnings)',
        'kir_top01': 'Income Rank – Probability Top 1%',
        'kir_top20': 'Income Rank – Probability Top 20%',
        'kir_24': 'Income Rank (age 24)',
        'kir_26': 'Income Rank (age 26)',
        'kir_29': 'Income Rank (age 29)',
        'hs': 'High School',
        'coll': 'College',
        'somecoll': 'Some College',
        'hours_wk': 'Weekly Hours Worked',
        'jail': 'Incarceration',
        'lpov_nbh': 'Low-Poverty Neighborhood',
        'proginc': 'Public Assistance',
        'staytract': 'Stayed in Tract',
        'teenbrth': 'Teen Birth',
        'wgflx_rk': 'Wage Rank',
        'working': 'Working'
        
        }

    df_long['outcome_label'] = df_long['outcome'].map(outcome_labels).fillna(df_long['outcome']) #fillna with original outcome label in case we missed danything in the dict

    df_long.sort_values(by=['state', 'county', 'cz', 'czname', 'outcome', 'race', 'gender', 'percentile'], inplace=True)
    
    # Save or return the processed DataFrame
    output_file = 'longcountyoutcomes.csv'
    df_long.to_csv(output_file, index=False)
    print(f"Data successfully transformed and saved to {output_file}")

    return df_long

In [7]:
file_path = 'county_outcomes.csv'
processed_df = process_county_state_dataset(file_path)

Data successfully transformed and saved to long_format_county_state_with_value_type.csv
